# SQLAlchemy ORM

## Overview
The **SQLAlchemy ORM** maps Python classes to database tables. You define models as Python classes, and SQLAlchemy generates SQL automatically for CRUD operations.

**ORM components:**
- **DeclarativeBase** — base class all models inherit from
- **Mapped / mapped_column** — 2.0-style column declarations with type annotations
- **Session** — the unit-of-work; tracks object changes; executes writes at `commit()`
- **relationship()** — declares associations between models (one-to-many, many-to-many)
- **selectinload / joinedload** — eager loading strategies to prevent N+1 queries

**SQLAlchemy 2.0 ORM style:**
```python
class Patient(Base):
    __tablename__ = "patients"
    patient_id: Mapped[str] = mapped_column(String, primary_key=True)
    first_name: Mapped[str] = mapped_column(String, nullable=False)
```

---

In [1]:
from sqlalchemy import create_engine, String, Float, Integer, ForeignKey
from sqlalchemy.orm import (DeclarativeBase, Mapped, mapped_column,
                             relationship, Session, selectinload)
from typing import List, Optional

engine = create_engine("sqlite:///:memory:", echo=False)

class Base(DeclarativeBase):
    pass

class Patient(Base):
    __tablename__ = "patients"
    patient_id : Mapped[str]           = mapped_column(String, primary_key=True)
    first_name : Mapped[str]           = mapped_column(String, nullable=False)
    last_name  : Mapped[str]           = mapped_column(String, nullable=False)
    province   : Mapped[str]           = mapped_column(String, nullable=False)
    sex        : Mapped[str]           = mapped_column(String, nullable=False)
    encounters : Mapped[List["Encounter"]] = relationship(back_populates="patient")

    def __repr__(self):
        return f"Patient({self.patient_id}, {self.first_name} {self.last_name})"

class Encounter(Base):
    __tablename__ = "encounters"
    enc_id     : Mapped[int]           = mapped_column(Integer, primary_key=True, autoincrement=True)
    patient_id : Mapped[str]           = mapped_column(ForeignKey("patients.patient_id"))
    enc_date   : Mapped[str]           = mapped_column(String, nullable=False)
    enc_type   : Mapped[str]           = mapped_column(String, nullable=False)
    cost       : Mapped[float]         = mapped_column(Float, default=0.0)
    patient    : Mapped["Patient"]     = relationship(back_populates="encounters")

    def __repr__(self):
        return f"Encounter({self.enc_id}, {self.enc_date}, ${self.cost:,.2f})"

Base.metadata.create_all(engine)
print("ORM models defined: Patient, Encounter")
print("Tables created:", list(Base.metadata.tables.keys()))


ORM models defined: Patient, Encounter
Tables created: ['patients', 'encounters']


---
## Session: inserting and querying

In [2]:
from sqlalchemy import create_engine, String, Float, Integer, ForeignKey
from sqlalchemy import select
from sqlalchemy.orm import (DeclarativeBase, Mapped, mapped_column,
                             relationship, Session, selectinload)
from typing import List

engine = create_engine("sqlite:///:memory:", echo=False)

class Base(DeclarativeBase): pass

class Patient(Base):
    __tablename__ = "patients"
    patient_id : Mapped[str]  = mapped_column(String, primary_key=True)
    first_name : Mapped[str]  = mapped_column(String, nullable=False)
    last_name  : Mapped[str]  = mapped_column(String, nullable=False)
    province   : Mapped[str]  = mapped_column(String, nullable=False)
    encounters : Mapped[List["Encounter"]] = relationship(back_populates="patient")
    def __repr__(self): return f"Patient({self.patient_id}, {self.first_name})"

class Encounter(Base):
    __tablename__ = "encounters"
    enc_id     : Mapped[int]   = mapped_column(Integer, primary_key=True, autoincrement=True)
    patient_id : Mapped[str]   = mapped_column(ForeignKey("patients.patient_id"))
    enc_date   : Mapped[str]   = mapped_column(String, nullable=False)
    enc_type   : Mapped[str]   = mapped_column(String, nullable=False)
    cost       : Mapped[float] = mapped_column(Float, default=0.0)
    patient    : Mapped["Patient"] = relationship(back_populates="encounters")

Base.metadata.create_all(engine)

# ── INSERT via session ────────────────────────────────────────────
with Session(engine) as session:
    aroha = Patient(patient_id="P001", first_name="Aroha", last_name="Ngata", province="NB")
    liam  = Patient(patient_id="P002", first_name="Liam",  last_name="Chen",  province="ON")
    session.add_all([aroha, liam])
    session.flush()   # write to DB within transaction; IDs are now available

    session.add_all([
        Encounter(patient_id="P001", enc_date="2023-04-10", enc_type="outpatient", cost=250.0),
        Encounter(patient_id="P001", enc_date="2023-07-15", enc_type="inpatient",  cost=4800.0),
        Encounter(patient_id="P002", enc_date="2023-03-01", enc_type="outpatient", cost=80.0),
    ])
    session.commit()
    print("Inserted 2 patients and 3 encounters")

# ── SELECT via session ────────────────────────────────────────────
with Session(engine) as session:
    # get() — fetch by primary key (most efficient)
    p = session.get(Patient, "P001")
    print(f"\nget() by PK: {p}")

    # select() — flexible querying
    stmt = select(Patient).where(Patient.province == "NB")
    nb_patients = session.execute(stmt).scalars().all()
    print(f"NB patients: {nb_patients}")

    # Eager load encounters to avoid N+1 queries
    stmt = (select(Patient)
            .options(selectinload(Patient.encounters))
            .order_by(Patient.last_name))
    patients = session.execute(stmt).scalars().all()
    print()
    for p in patients:
        total = sum(e.cost for e in p.encounters)
        print(f"  {p.first_name} {p.last_name}  "
              f"encounters={len(p.encounters)}  total=${total:,.2f}")


Inserted 2 patients and 3 encounters

get() by PK: Patient(P001, Aroha)
NB patients: [Patient(P001, Aroha)]

  Liam Chen  encounters=1  total=$80.00
  Aroha Ngata  encounters=2  total=$5,050.00


---
## Update, delete, and session lifecycle

In [3]:
from sqlalchemy import create_engine, String, Float, Integer, ForeignKey, update, delete
from sqlalchemy import select
from sqlalchemy.orm import (DeclarativeBase, Mapped, mapped_column,
                             relationship, Session)
from typing import List

engine = create_engine("sqlite:///:memory:", echo=False)

class Base(DeclarativeBase): pass
class Patient(Base):
    __tablename__ = "patients"
    patient_id: Mapped[str]   = mapped_column(String, primary_key=True)
    first_name: Mapped[str]   = mapped_column(String, nullable=False)
    last_name:  Mapped[str]   = mapped_column(String, nullable=False)
    province:   Mapped[str]   = mapped_column(String, nullable=False)
    encounters: Mapped[List["Encounter"]] = relationship(back_populates="patient", cascade="all, delete-orphan")
class Encounter(Base):
    __tablename__ = "encounters"
    enc_id:     Mapped[int]   = mapped_column(Integer, primary_key=True, autoincrement=True)
    patient_id: Mapped[str]   = mapped_column(ForeignKey("patients.patient_id"))
    enc_date:   Mapped[str]   = mapped_column(String, nullable=False)
    enc_type:   Mapped[str]   = mapped_column(String, nullable=False)
    cost:       Mapped[float] = mapped_column(Float, default=0.0)
    patient:    Mapped["Patient"] = relationship(back_populates="encounters")

Base.metadata.create_all(engine)
with Session(engine) as session:
    session.add_all([
        Patient(patient_id="P001", first_name="Aroha", last_name="Ngata", province="NB"),
        Patient(patient_id="P002", first_name="Liam",  last_name="Chen",  province="ON"),
    ])
    session.add_all([
        Encounter(patient_id="P001", enc_date="2023-04-10", enc_type="outpatient", cost=250.0),
        Encounter(patient_id="P001", enc_date="2023-07-15", enc_type="inpatient",  cost=4800.0),
    ])
    session.commit()

# ── Update: via object attribute assignment ──────────────────────
with Session(engine) as session:
    p = session.get(Patient, "P001")
    p.province = "NS"               # just assign; session tracks the change
    session.commit()
    print("After update — province:", session.get(Patient, "P001").province)

# ── Bulk UPDATE with WHERE (Core-style, 2.0) ─────────────────────
with Session(engine) as session:
    session.execute(
        update(Encounter)
        .where(Encounter.enc_type == "outpatient")
        .values(cost=Encounter.cost * 1.05)    # 5% cost increase
    )
    session.commit()
    enc = session.execute(
        select(Encounter).where(Encounter.enc_type == "outpatient")
    ).scalars().first()
    print(f"After bulk update — outpatient cost: ${enc.cost:,.2f}")

# ── Delete: cascade deletes child encounters ─────────────────────
with Session(engine) as session:
    p = session.get(Patient, "P001")
    session.delete(p)   # cascade="all, delete-orphan" removes encounters too
    session.commit()
    remaining = session.execute(select(Patient)).scalars().all()
    enc_remaining = session.execute(select(Encounter)).scalars().all()
    print(f"After delete — patients={len(remaining)}, encounters={len(enc_remaining)}")


After update — province: NS
After bulk update — outpatient cost: $262.50
After delete — patients=1, encounters=0


---
## Common Pitfalls

**1. Forgetting to call session.commit() after writes**
Changes added to a Session are tracked but not written to the database until `session.commit()`. Using `with Session(engine) as session:` does NOT auto-commit -- you must call `session.commit()` explicitly. Use `session.flush()` to write to the DB within the transaction without committing (useful when you need the auto-generated ID before committing).

**2. The N+1 query problem with lazy-loaded relationships**
Iterating over a list of patients and accessing `patient.encounters` inside the loop fires one SQL query per patient if `encounters` is lazy-loaded (the default). With 100 patients, that is 101 queries. Use `joinedload` or `selectinload` in the query to fetch related records in 1-2 queries: `session.query(Patient).options(selectinload(Patient.encounters)).all()`.

**3. Modifying objects after session.expunge() or session.close()**
After a Session closes, ORM objects become detached. Accessing lazy-loaded relationships on a detached object raises `DetachedInstanceError`. Either load all required data eagerly before closing the session, or re-attach the object with `session.merge(obj)`.

**4. Using session.query() in SQLAlchemy 2.0**
`session.query(Patient)` is the legacy 1.x API. In SQLAlchemy 2.0, use `session.execute(select(Patient)).scalars().all()`. Both work in 2.0 but the new style is consistent with Core and is the future-facing approach.

**5. Creating a new engine per request in web applications**
The Engine holds the connection pool. Creating a new Engine per HTTP request bypasses pooling and opens a new database connection for every request -- destroying performance. Create the Engine once at application startup and share it across all requests.

**6. Not using a scoped_session in multi-threaded applications**
A plain `Session` is not thread-safe. In Flask/FastAPI/Django, each thread or async context needs its own Session. Use `sessionmaker` with `scoped_session` (Flask) or SQLAlchemy's async Session (FastAPI) to ensure each request gets its own isolated Session.


---
*sql_methods_library - Samantha McGarrigle*